# M0.1 · What an agent actually is

**Module 0 — the shared core → The Shared Core**  ·  *Both directions*

---

**Risk.** "The model did it" is treated as a root cause, so the real control gap is never found.

**Control.** Separate the three planes; locate autonomy in what the model's output is allowed to trigger.

**This lab.** Show that autonomy lives in the action plane, not the model.

| | |
|---|---|
| Open-source tooling | Ollama, llama.cpp |
| Open-weight models | GLM-4.6, Llama 3.3 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("M0.1"))

Three planes, one rule: **the model only ever writes on the decision plane**. If state changed, something on the control plane let a proposal through. That is why "the model did it" is never a root cause — it names the only component that structurally cannot be one.

Take the same capability list and ask which plane each tool sits on. The answer comes from what the tool *can do*, not from what it is called.

In [ ]:
from cybercommons import planes

tools = [
    planes.Tool("search_docs"),                                    # read
    planes.Tool("read_file"),                                      # read
    planes.Tool("post_comment", writes=True, scope="project"),
    planes.Tool("merge_pr",     writes=True, scope="project", reversible=False),
    planes.Tool("deploy_prod",  writes=True, scope="org",     reversible=False),
]

for t in tools:
    print(f"{t.name:14s} {t.plane:9s} writes={str(t.writes):5s} scope={t.scope}")

Now the same question the other way round: given a manifest, what can one unreviewed action actually cost? This is the number A1.4 turns into a design metric.

In [ ]:
copilot = planes.Manifest("copilot", tools[:2], rung="L1")
agent   = planes.Manifest("remediation-agent", tools, rung="L2")

for m in (copilot, agent):
    b = m.blast_radius()
    print(f"\n{m.agent}  (claims {m.rung})")
    print("  planes:", {k: v for k, v in m.by_plane().items() if v})
    print("  blast radius:", b["total"], b["per_tool"])
    for problem in m.rung_check():
        print("  ⚠", problem)

The bare model and the copilot cannot change state at all. The agent can — and it claims a rung its controls do not support. That gap, not the model's cleverness, is the security story.

### Expect

The first two configurations have a blast radius of 0 — they hold no state-changing tool. The agent scores non-zero and `rung_check()` reports that it claims L2 (approve every call) while every writer is ungated.

### Your turn

Add `planes.Tool("rotate_secrets", writes=True, scope="org", reversible=False)` to the agent. Predict the new blast radius before you run it, then gate it with `approval_required` and watch the number fall to where it was.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/M0.1.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*